# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted)

## Get entities LLM

In [ ]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_PreLabel.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]


### no boost

In [ ]:
#%%
# Wrap original extract_llm_entities with a progress bar (no need to modify .py)
def extract_llm_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="LLM Prediction"):
        result = extract_llm_entities([sent])[0]
        results.append(result)
    return results

In [2]:
#%%
# Run prediction with progress bar
pred_results = extract_llm_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 445/445 [28:07<00:00,  3.79s/it]


### boost

In [2]:
from tqdm import tqdm

BATCH_SIZE = 200  # 可視 rate limit 調 100~500

def extract_llm_entities_in_batches(sentences, batch_size=BATCH_SIZE):
    results = []
    n = len(sentences)
    for start in tqdm(range(0, n, batch_size), desc="LLM Prediction"):
        batch = sentences[start:start+batch_size]
        # 一次丟一大批，讓 extract_llm_entities 內部開並行
        results.extend(extract_llm_entities(batch))
    return results

# 使用
pred_results = extract_llm_entities_in_batches(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 3/3 [02:21<00:00, 47.03s/it]


### evaluate

In [3]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [4]:
#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.6103983794733289
Macro F1: 0.5016426737227713
Weighted F1: 0.6114972944510506

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.66      0.64      0.65        77
           Interest Rates       0.75      0.55      0.64        49
                Inflation       0.95      0.92      0.93        85
               Employment       0.74      0.79      0.76        33
             Unemployment       0.89      1.00      0.94         8
                      GDP       0.48      0.46      0.47        26
                    Trade       0.75      1.00      0.86         6
                 Congress       0.00      0.00      0.00         0
          Monetary Policy       0.59      0.70      0.64        70
      Financial Stability       0.00      0.00      0.00         0
          Price Stability       0.64      0.41      0.50        17
Regulatory Implementation       0.00      0.00      0.00         2
              

# Evaluation Sentiment Analysis

In [3]:
# eval_sentiment_both.py
# Evaluate sentiment scoring (nlp + llm) on the holdout set using gold entities.
# No aliasing/canonicalization; duplicates handled by per-sentence position index.

import ast
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    classification_report,
)

from get_sentiment_nlp import extract_nlp_sentiment
from get_sentiment_llm import extract_llm_sentiment

HOLDOUT_CSV = "holdout_eval_set.csv"

# ---------------------------
# parse the "Entities" cell -> ordered list[(entity, score)]
def parse_entities_cell(s):
    try:
        raw = ast.literal_eval(s)
    except Exception:
        return []
    out = []
    for tup in raw or []:
        if not isinstance(tup, (list, tuple)) or len(tup) < 2:
            continue
        ent, scr = tup[0], tup[1]
        if ent in (None, "") or scr in ("", None):
            continue
        try:
            out.append((str(ent), float(scr)))
        except Exception:
            pass
    return out

# ---------------------------
# flatten results -> DataFrame(sentence, pos, entity, pred_score)
def flatten_pred_struct(results):
    rows = []
    for item in results:
        s = item["sentence"]
        for i, ent in enumerate(item["entities"]):
            rows.append({"sentence": s, "pos": i, "entity": ent["name"], "pred_score": ent["sentiment"]})
    return pd.DataFrame(rows)

# ---------------------------
# compute and print metrics (overall + per-entity) including MAE and baselines
def compute_and_print_metrics(name, merged, score_to_label):
    def to_label(x):
        try:
            return score_to_label[float(x)]
        except Exception:
            return None

    df = merged.copy()
    df = df[(df["pred_score"] != "") & (~df["pred_score"].isna())].copy()
    if df.empty:
        print(f"\n== {name} ==\nNo comparable predictions after alignment.")
        return

    # classification labels
    df["y_true"] = df["true_score"].apply(to_label)
    df["y_pred"] = df["pred_score"].apply(to_label)
    df = df.dropna(subset=["y_true", "y_pred"]).astype({"y_true": int, "y_pred": int})
    if df.empty:
        print(f"\n== {name} ==\nNo valid label pairs after mapping.")
        return

    labels_sorted = sorted(set(score_to_label.values()))
    macro_f1 = f1_score(df["y_true"], df["y_pred"], average="macro", labels=labels_sorted)
    micro_f1 = f1_score(df["y_true"], df["y_pred"], average="micro", labels=labels_sorted)
    weighted_f1 = f1_score(df["y_true"], df["y_pred"], average="weighted", labels=labels_sorted)
    acc = accuracy_score(df["y_true"], df["y_pred"])

    macro_precision = precision_score(df["y_true"], df["y_pred"], average="macro", labels=labels_sorted, zero_division=0)
    macro_recall = recall_score(df["y_true"], df["y_pred"], average="macro", labels=labels_sorted, zero_division=0)
    micro_precision = precision_score(df["y_true"], df["y_pred"], average="micro", labels=labels_sorted, zero_division=0)
    micro_recall = recall_score(df["y_true"], df["y_pred"], average="micro", labels=labels_sorted, zero_division=0)
    weighted_precision = precision_score(df["y_true"], df["y_pred"], average="weighted", labels=labels_sorted, zero_division=0)
    weighted_recall = recall_score(df["y_true"], df["y_pred"], average="weighted", labels=labels_sorted, zero_division=0)

    # MAE for model predictions (work in raw score space)
    y_true = df["true_score"].astype(float).to_numpy()
    y_pred = df["pred_score"].astype(float).to_numpy()
    mae = float(np.mean(np.abs(y_pred - y_true)))

    # --- Baselines (computed from THIS eval set) ---
    # Majority-class (mode) baseline for MAE
    mode_score = float(pd.Series(y_true).mode().iloc[0])  # if tie, picks first mode
    mae_mode = float(np.mean(np.abs(mode_score - y_true)))

    # MAE-optimal constant baseline (median)
    median_score = float(np.median(y_true))
    mae_median = float(np.mean(np.abs(median_score - y_true)))

    print(f"\n== {name} ==  (n={len(df)})")
    print(f"Accuracy              : {acc:.4f}")
    print(f"Macro F1              : {macro_f1:.4f}")
    print(f"Macro Precision       : {macro_precision:.4f}")
    print(f"Macro Recall          : {macro_recall:.4f}")
    print(f"Micro F1              : {micro_f1:.4f}")
    print(f"Micro Precision       : {micro_precision:.4f}")
    print(f"Micro Recall          : {micro_recall:.4f}")
    print(f"Weighted F1           : {weighted_f1:.4f}")
    print(f"Weighted Precision    : {weighted_precision:.4f}")
    print(f"Weighted Recall       : {weighted_recall:.4f}")
    print(f"MAE (model)           : {mae:.4f}")
    print(f"MAE (majority baseline: predict {mode_score:+.2f}) : {mae_mode:.4f}  "
          f"(Δ vs model = {mae_mode - mae:+.4f})")
    print(f"MAE (median   baseline: predict {median_score:+.2f}) : {mae_median:.4f}  "
          f"(Δ vs model = {mae_median - mae:+.4f})")

    # per-entity breakdown (classification view)
    print("\n-- Per-entity (macro F1 / acc / precision / recall / support) --")
    per_rows = []
    for ent, sub in df.groupby("entity"):
        f1_ent = f1_score(sub["y_true"], sub["y_pred"], average="macro", labels=labels_sorted, zero_division=0)
        acc_ent = accuracy_score(sub["y_true"], sub["y_pred"])
        prec_ent = precision_score(sub["y_true"], sub["y_pred"], average="macro", labels=labels_sorted, zero_division=0)
        rec_ent = recall_score(sub["y_true"], sub["y_pred"], average="macro", labels=labels_sorted, zero_division=0)
        per_rows.append((ent, len(sub), f1_ent, acc_ent, prec_ent, rec_ent))
    per_rows.sort(key=lambda x: (-x[1], -x[2]))
    for ent, n, f1e, acce, prece, recce in per_rows:
        print(f"{ent:<28} macroF1={f1e:.3f}  acc={acce:.3f}  prec={prece:.3f}  rec={recce:.3f}  n={n}")

    # Optionally, print a full classification report
    print("\nClassification Report:")
    print(classification_report(
        df["y_true"], df["y_pred"],
        labels=labels_sorted,
        zero_division=0
    ))

# ---------------------------
# main
if __name__ == "__main__":
    with open("score_to_label.json", "r") as fp:
        s2l = json.load(fp)
    score_to_label = {float(k): int(v) for k, v in s2l.items()}

    df = pd.read_csv(HOLDOUT_CSV)
    by_sent = defaultdict(list)
    for _, r in df.iterrows():
        s = r.get("Sentence", "")
        ents = parse_entities_cell(r.get("Entities", "[]"))
        if ents:
            by_sent[s].extend(ents)

    gold_rows = []
    for s, ents in by_sent.items():
        for i, (e, sc) in enumerate(ents):
            gold_rows.append({"sentence": s, "pos": i, "entity": e, "true_score": sc})
    gold = pd.DataFrame(gold_rows)
    if gold.empty:
        raise ValueError("Holdout has no valid (sentence, entity, score) rows.")

    items_llm = [{"sentence": s, "entities": [{"name": e} for e, _ in ents]} for s, ents in by_sent.items()]
    items_nlp = [(s, [e for e, _ in ents]) for s, ents in by_sent.items()]

    preds_llm = extract_llm_sentiment(items_llm)
    preds_nlp = extract_nlp_sentiment(items_nlp)

    df_llm = flatten_pred_struct(preds_llm)
    df_nlp = flatten_pred_struct(preds_nlp)

    merged_llm = gold.merge(df_llm, on=["sentence", "pos", "entity"], how="left")
    merged_nlp = gold.merge(df_nlp, on=["sentence", "pos", "entity"], how="left")

    compute_and_print_metrics("LLM", merged_llm, score_to_label)
    compute_and_print_metrics("NLP", merged_nlp, score_to_label)


2025-08-26 11:36:31,667 | INFO | [extract] received items=288 | with_entities=288 | skipped=0
2025-08-26 11:36:31,667 | INFO | [extract] batching | chunks=15 | chunk_size≈20
2025-08-26 11:36:31,987 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:31,988 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:31,988 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:32,397 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:48,167 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:49,677 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:52,161 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 11:36:55,744 | INFO | HTTP Requ


== LLM ==  (n=776)
Accuracy              : 0.2474
Macro F1              : 0.1333
Macro Precision       : 0.1148
Macro Recall          : 0.2017
Micro F1              : 0.2474
Micro Precision       : 0.2474
Micro Recall          : 0.2474
Weighted F1           : 0.1821
Weighted Precision    : 0.1754
Weighted Recall       : 0.2474
MAE (model)           : 0.3689
MAE (majority baseline: predict +0.33) : 0.3506  (Δ vs model = -0.0183)
MAE (median   baseline: predict +0.33) : 0.3506  (Δ vs model = -0.0183)

-- Per-entity (macro F1 / acc / precision / recall / support) --
Economic Outlook             macroF1=0.105  acc=0.200  prec=0.089  rec=0.161  n=115
Federal Reserve              macroF1=0.058  acc=0.255  prec=0.037  rec=0.143  n=102
Inflation                    macroF1=0.125  acc=0.244  prec=0.115  rec=0.193  n=86
Monetary Policy              macroF1=0.060  acc=0.209  prec=0.061  rec=0.138  n=67
Interest Rates               macroF1=0.090  acc=0.179  prec=0.134  rec=0.153  n=39
Employment  